# polygon → MOC → shard → coincident waveforms

The waveform half of the minimal read stack. Same two libraries, same polygon,
same public store, zero credentials — but instead of the 3-D scatter, this
notebook does the **cell-level join**: one GEDI o18 footprint against the 2×2
ATL03 o19 cells beneath it, both reconstructed from their stored t-digests as
densities on a shared elevation axis.

Its sibling is [`hhdc_viewer.ipynb`](hhdc_viewer.ipynb), which takes the same
polygon and the same stores to a rotatable paired 3-D view and numpy tensors.
The two are separate notebooks on purpose: that one needs `%matplotlib widget`
for the 3-D view, this one needs `%matplotlib inline`, and the two backends
collide in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipywidgets
%matplotlib inline

import time

import moczarr as mz
from mortie import moc

# The drawing lives in viewers.py beside this notebook, so the cells below stay
# about the READ path. It is also where the one zagg import lives: moczarr
# imports the t-digest algebra rather than vendoring it (moczarr issue #19),
# which is what the `moczarr[zagg]` extra carries.
from viewers import BLOCK_ORDER, waveform_view

STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

ASIDE, GSIDE = 128, 64  # cells across an o12 block: 2**(19-12) and 2**(18-12)
N_BINS, RES = 256, 1.0  # shared z grid for the paired tensors

## One polygon in, covered shards out

Replace the polygon with any area within California or a NEON AOP site; the
cell tests whether each store actually covers it. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [  # a ~4 km box on the SERC tract
                        [-76.56, 38.87],
                        [-76.50, 38.87],
                        [-76.50, 38.91],
                        [-76.56, 38.91],
                        [-76.56, 38.87],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)
shards

## Paired tensors — one shard, both sensors

`read_tensors` yields one `(tensor, mask, (offset, gain), block)` per populated
o12 block. Reading both sensors on the same z grid is what makes the two
comparable; the blocks they share are the candidates for a join.

In [ ]:
stores = {name: mz.open_leaf(root, shards[0], **S3) for name, (root, _f) in STORES.items()}

t0 = time.perf_counter()
blocks = {
    name: {
        b[3]: b
        for b in mz.read_tensors(
            stores[name],
            field,
            n_bins=N_BINS,
            resolution=RES,
            block_order=BLOCK_ORDER,
            fit="degrade_resolution",
        )
    }
    for name, (_root, field) in STORES.items()
}
print(
    f"paired tensors in {time.perf_counter() - t0:.1f}s — "
    + ", ".join(f"{len(v)} {k} blocks" for k, v in blocks.items())
)

# Fold ATL03's o19 footprint down to GEDI's o18 grid, then keep the cells both
# sensors actually populate. SORTED by joint-cell count, densest first, so the
# first pick has real data on both sides.
pairs = []
for w, (gt, _gm, _gz, _) in blocks["gedi"].items():
    if w not in blocks["atl03"]:
        continue
    A2 = blocks["atl03"][w][0].sum(axis=2).reshape(GSIDE, 2, GSIDE, 2).sum(axis=(1, 3))
    G2 = gt.sum(axis=2)
    joint = (A2 > 0) & (G2 > 0)
    if joint.any():
        pairs.append((w, joint, A2, G2))
pairs.sort(key=lambda p: -int(p[1].sum()))

for w, j, A2, G2 in pairs[:8]:
    print(
        f"  {mz.morton_decimal(w)}  {int(j.sum()):4,} joint o18 cells   "
        f"{int(A2[j].sum()):9,} atl03 photons   {int(G2[j].sum()):9,} gedi pe"
    )

## Coincident waveforms

One GEDI o18 cell against the 2×2 ATL03 o19 cells under it, both read straight
from their stored digests — no tensor in the loop. `cell_index` turns a
chunk-local `(row, col)` into the global cells-axis index `read_cell` wants;
GEDI's chunk grid puts one o12 block in one chunk, while ATL03's chunks sit at
o13, so the ATL03 side resolves which of the block's four o13 children a cell
falls in before addressing it.

What makes a pick *coincident* is the joint mask above — both sensors
populated. Within that, the slider orders cells by the **weaker** member. ATL03
photons and GEDI photoelectrons are not commensurate (the pe run two to three
orders of magnitude higher), so a raw `min(photons, pe)` would just be the
ATL03 count; each side is scaled by its own maximum over the block's joint
cells first, and the min of those ranks by whichever sensor is *relatively*
weaker.

In [ ]:
FIELDS = {name: field for name, (_root, field) in STORES.items()}
waveform_view(stores, FIELDS, blocks, pairs, shards[0], gside=GSIDE)

Two libraries, one polygon — coverage, shards, paired tensors, and the
cell-level waveform join, anonymously against public S3.